In [4]:
import os
import sys

# Pfad zum übergeordneten Verzeichnis hinzufügen (damit themealdb_client gefunden wird)
sys.path.insert(0, os.path.abspath('..'))

from themealdb_client import TheMealDBClient
import json
import traceback

# Test für get_all_ingredients() Funktion

# Client initialisieren
client = TheMealDBClient()
def get_all_ingredients():
    try:
        ingredients = client.get_all_ingredients()
        ingredients = [ingredient['strIngredient'] for ingredient in ingredients if 'strIngredient' in ingredient]
        print(f"Anzahl Zutaten gefunden: {len(ingredients)}")
        print("Beispiel-Zutaten:")
        for ing in ingredients[:10]:  # Zeige die ersten 10 Zutaten
            print(f"- {ing}")
        return ingredients
    except Exception as e:
        print("Fehler beim Abrufen der Zutaten:")
        traceback.print_exc()
        return []
ALL_INGREDIENTS = get_all_ingredients()

Anzahl Zutaten gefunden: 877
Beispiel-Zutaten:
- Chicken
- Salmon
- Beef
- Pork
- Avocado
- Apple Cider Vinegar
- Asparagus
- Aubergine
- Baby Plum Tomatoes
- Bacon


In [7]:
import spacy
from collections import defaultdict

try:
    nlp = spacy.load("en_core_web_md")
except:
    nlp = spacy.load("en_core_web_sm")

# 1. NOISE WORDS (Ignorable)
# Diese Wörter dürfen im User-Input stehen, aber in der DB fehlen (und umgekehrt).
# Sie ändern die Identität nicht.
IGNORABLE_TOKENS = {
    "chopped", "sliced", "diced", "minced", "fresh", "raw", "organic", 
    "large", "small", "whole", "halves", "pieces", "leaf", "leaves", 
    "seed", "seeds", "ground", "crushed"
}

# 2. CRITICAL MODIFIERS (Barrier)
# Diese Wörter darf die Datenbank NICHT hinzufügen, wenn der User sie nicht genannt hat.
CRITICAL_MODIFIERS = {
    "oil", "sauce", "paste", "puree", "juice", "extract", 
    "butter", "flour", "liver", "heart", "stock", "broth", "vinegar", "wine"
}

MANUAL_CORRECTIONS = {"leaves": "leaf", "leave": "leaf"}

class PreciseIngredientMapper:
    def __init__(self, all_ingredients_list):
        self.ingredient_map = defaultdict(set)
        self._build_index(all_ingredients_list)

    def _get_tokens(self, text):
        doc = nlp(text.lower())
        tokens = set()
        for token in doc:
            if not token.is_stop and not token.is_punct:
                lemma = MANUAL_CORRECTIONS.get(token.text, token.lemma_)
                if lemma.endswith("s") and len(lemma) > 3:
                    tokens.add(lemma[:-1])
                else:
                    tokens.add(lemma)
        return tokens

    def _build_index(self, ingredients):
        for original_name in ingredients:
            tokens = self._get_tokens(original_name)
            # Wir speichern unter jedem Token, damit wir schnell filtern können
            for token in tokens:
                self.ingredient_map[token].add(original_name)

    def get_similar_ingredients(self, user_search):
        user_tokens = self._get_tokens(user_search)
        if not user_tokens: return []

        # A. Essentielle User-Tokens identifizieren
        # Alles was NICHT 'Noise' ist, ist eine harte Anforderung.
        required_user_tokens = user_tokens - IGNORABLE_TOKENS
        
        # Performance-Trick: Wir holen Kandidaten basierend auf dem ersten Token
        # (Egal welchem, wir prüfen gleich eh alle streng)
        candidates = set()
        for token in user_tokens:
            candidates.update(self.ingredient_map.get(token, []))
        
        final_matches = []
        for db_ingredient in candidates:
            db_tokens = self._get_tokens(db_ingredient)
            
            # CHECK 1: Sind alle ESSENTIELLEN User-Wörter vorhanden?
            # Wenn User "Walnut Oil" will -> "Walnut" und "Oil" sind essentiell.
            # "Olive Oil" hat "Walnut" nicht -> Raus!
            if not required_user_tokens.issubset(db_tokens):
                continue

            # CHECK 2: Hat die DB verbotene Wörter hinzugefügt?
            # Was hat die DB mehr als der User?
            # User: "Walnut", DB: "Walnut Oil" -> diff is "Oil"
            extra_db_words = db_tokens - user_tokens
            
            # Wenn eines der extra Wörter auf der Blacklist steht -> Raus!
            if not extra_db_words.isdisjoint(CRITICAL_MODIFIERS):
                continue
                
            final_matches.append(db_ingredient)
            
        return final_matches

# # --- HARTE TESTS ---
# ALL_INGREDIENTS = [
#     "Walnut", "Walnuts", "Chopped Walnuts", "Walnut Oil", "Olive Oil", 
#     "Basil", "Basil Leaves", "Tomato", "Tomato Sauce"
# ]

mapper = PreciseIngredientMapper(ALL_INGREDIENTS)

def test(query):
    print(f"Suche: '{query:<15}' -> Gefunden: {mapper.get_similar_ingredients(query)}")

print("--- Test: Spezifisches Öl ---")
test("Walnut Oil") 
# ✅ Findet: ['Walnut Oil']
# ❌ Ignoriert: 'Olive Oil' (Weil 'Walnut' fehlt)
# ❌ Ignoriert: 'Walnuts' (Weil 'Oil' fehlt)

print("\n--- Test: Allgemeine Nuss ---")
test("Walnut")
# ✅ Findet: ['Walnut', 'Walnuts', 'Chopped Walnuts']
# ❌ Ignoriert: 'Walnut Oil' (Weil 'Oil' extra ist und kritisch)

print("\n--- Test: Ignorables (Leaves/Chopped) ---")
test("Basil")
# ✅ Findet: ['Basil', 'Basil Leaves'] (Leaves ist ignorable Zusatz)

test("Chopped Walnuts")
# ✅ Findet: ['Walnut', 'Walnuts', 'Chopped Walnuts'] 
# ('Chopped' ist required_user_tokens = empty oder ignoriert, weil es in IGNORABLE steht)

--- Test: Spezifisches Öl ---
Suche: 'Walnut Oil     ' -> Gefunden: []

--- Test: Allgemeine Nuss ---
Suche: 'Walnut         ' -> Gefunden: ['Walnuts']

--- Test: Ignorables (Leaves/Chopped) ---
Suche: 'Basil          ' -> Gefunden: ['Basil Leaves', 'Basil', 'Fresh Basil']
Suche: 'Chopped Walnuts' -> Gefunden: []


In [ ]:
import spacy
from collections import defaultdict

try:
    nlp = spacy.load("en_core_web_md")
except:
    nlp = spacy.load("en_core_web_sm")

# 1. NOISE (Ignorieren wir komplett)
NOISE_WORDS = {
    "chopped", "sliced", "diced", "minced", "fresh", "raw", "organic", 
    "large", "small", "whole", "halves", "pieces", "ground", "crushed", 
    "leaves", "leaf", "seed", "seeds", "nuts", "dark", "white", "yolks",
    "free-range", "beat", "stoned"  # Leaves/Seeds hier als Noise, da oft optional
}

# 2. STATES / FORMS (Definieren eine NEUE Gruppe)
# Wenn eines dieser Wörter auftaucht, ändert sich die Bedeutung fundamental.
# Das sind die "Zusätze", die den Kern verändern.
# FORM_MODIFIERS = {
#     "oil", "sauce", "paste", "puree", "juice", "extract", 
#     "butter", "flour", "liver", "heart", "stock", "broth", "vinegar", "soup"
# }
FORM_MODIFIERS = {
    # "sauce",  "puree", "juice", "extract", 
    # "butter", "flour", "liver", "heart", "stock", "broth",  "soup"
}

MANUAL_CORRECTIONS = {"leaves": "leaf", "leave": "leaf", "nuts": "nut", "yolks": "yolk"}

class SemanticGrouper:
    def __init__(self, all_ingredients_list):
        # Der Key ist jetzt ein Tuple: (Hauptzutat, Verarbeitungsform)
        # z.B. ('walnut', None) oder ('walnut', 'oil')
        self.groups = defaultdict(list)
        self._build_index(all_ingredients_list)

    def _analyze(self, text):
        """
        Zerlegt den Text in (Entity, Form).
        """
        doc = nlp(text.lower())
        
        entities = []
        found_form = None # Wir gehen davon aus, dass es nur EINE Haupt-Form gibt (z.B. Oil)

        for token in doc:
            if token.is_stop or token.is_punct:
                continue
            
            lemma = MANUAL_CORRECTIONS.get(token.text, token.lemma_)
            
            # S-Hack
            if lemma.endswith("s") and len(lemma) > 3:
                lemma = lemma[:-1]

            # 1. Ist es Noise? -> Wegwerfen
            if lemma in NOISE_WORDS:
                continue
            
            # 2. Ist es eine Form (Modifier)? -> Speichern
            if lemma in FORM_MODIFIERS:
                found_form = lemma
                continue
            
            # 3. Wenn es weder Noise noch Form ist, muss es die Substanz sein
            entities.append(lemma)
        
        # Sortieren der Entities (z.B. "Plum Tomato" -> "plum tomato")
        entities.sort()
        entity_str = " ".join(entities)
        
        # Rückgabe des Tuples (Der "Fingerabdruck" der Zutat)
        return (entity_str, found_form)

    def _build_index(self, ingredients):
        for raw_name in ingredients:
            key = self._analyze(raw_name)
            if key[0]: # Nur hinzufügen, wenn eine Substanz gefunden wurde
                self.groups[key].append(raw_name)

    def get_similar_ingredients(self, user_search):
        search_key = self._analyze(user_search)
        search_entity, search_form = search_key
        
        print(f"DEBUG: Suche nach Substanz='{search_entity}' | Form='{search_form}'")
        
        # Logik: Exakter Match auf die Gruppe
        # Wer "Walnut Oil" sucht, kriegt nur die Gruppe ('walnut', 'oil')
        if search_key in self.groups:
            return self.groups[search_key]
        
        # FALLBACK (Optional):
        # Wenn User nur "Walnut" sucht (Form=None), 
        # wollen wir vielleicht NUR die reine Nuss, oder?
        # Ja -> return [] (Strict)
        
        return []


grouper = SemanticGrouper(ALL_INGREDIENTS)

print("\n--- Datenbank Index ---")
for index, (key, vals) in enumerate(grouper.groups.items()):
    print(f"Gruppe {key}: {vals}")
    if index >= 5:
        print("...")  # Nur die ersten 5 Gruppen anzeigen
        break




--- Datenbank Index ---
Gruppe ('chicken', None): ['Chicken']
Gruppe ('salmon', None): ['Salmon']
Gruppe ('beef', None): ['Beef', 'Ground Beef']
Gruppe ('pork', None): ['Pork', 'Ground Pork']
Gruppe ('avocado', None): ['Avocado']
Gruppe ('apple cider vinegar', None): ['Apple Cider Vinegar']
...


In [64]:
for i in "Plum Tomatoes","Basmati Rice","Rice","Vinegar","White Vinegar","White Flour","Tomato", "Tomato Puree","Tomato Sauce","Chicken","Chicken Liver","Chicken Breast":
    print(grouper.get_similar_ingredients(i))

DEBUG: Suche nach Substanz='plum tomato' | Form='None'
['Plum Tomatoes']
DEBUG: Suche nach Substanz='basmati rice' | Form='None'
['Basmati Rice']
DEBUG: Suche nach Substanz='rice' | Form='None'
['Rice']
DEBUG: Suche nach Substanz='vinegar' | Form='None'
['Vinegar', 'White Vinegar']
DEBUG: Suche nach Substanz='vinegar' | Form='None'
['Vinegar', 'White Vinegar']
DEBUG: Suche nach Substanz='flour' | Form='None'
['Flour', 'White Flour']
DEBUG: Suche nach Substanz='tomato' | Form='None'
['Diced Tomatoes', 'Tomatoes', 'Tomato']
DEBUG: Suche nach Substanz='puree tomato' | Form='None'
['Tomato Puree']
DEBUG: Suche nach Substanz='sauce tomato' | Form='None'
['Tomato Sauce']
DEBUG: Suche nach Substanz='chicken' | Form='None'
['Chicken']
DEBUG: Suche nach Substanz='chicken liver' | Form='None'
['Chicken Liver']
DEBUG: Suche nach Substanz='breast chicken' | Form='None'
['Chicken Breast', 'Chicken Breasts']


In [63]:
fehler = []
normal = []
for ingredient in ALL_INGREDIENTS:
    if "Vinegar".lower() not in ingredient.lower():
        continue
    group = grouper.get_similar_ingredients(ingredient)
    if len(group) == 0:
        fehler.append(ingredient)
    else:
        normal.append((ingredient, group))
        
normal.sort(key=lambda x: len(x[1]), reverse=True)

for list_, name in [(fehler, "FEHLER"), (normal, "NORMAL")]:
    print(f"\n--- {name} ---")
    if list_:
        for item in list_:
            print(item)
    

DEBUG: Suche nach Substanz='apple cider vinegar' | Form='None'
DEBUG: Suche nach Substanz='balsamic vinegar' | Form='None'
DEBUG: Suche nach Substanz='vinegar' | Form='None'
DEBUG: Suche nach Substanz='vinegar' | Form='None'
DEBUG: Suche nach Substanz='red vinegar wine' | Form='None'
DEBUG: Suche nach Substanz='malt vinegar' | Form='None'
DEBUG: Suche nach Substanz='rice vinegar' | Form='None'
DEBUG: Suche nach Substanz='vinegar wine' | Form='None'
DEBUG: Suche nach Substanz='cider vinegar' | Form='None'
DEBUG: Suche nach Substanz='sherry vinegar' | Form='None'
DEBUG: Suche nach Substanz='vegan vinegar wine' | Form='None'
DEBUG: Suche nach Substanz='rice season vinegar' | Form='None'

--- FEHLER ---

--- NORMAL ---
('Vinegar', ['Vinegar', 'White Vinegar'])
('White Vinegar', ['Vinegar', 'White Vinegar'])
('Apple Cider Vinegar', ['Apple Cider Vinegar'])
('Balsamic Vinegar', ['Balsamic Vinegar'])
('Red Wine Vinegar', ['Red Wine Vinegar'])
('Malt Vinegar', ['Malt Vinegar'])
('Rice Vinegar'

In [58]:
fehler = []
normal = []
for ingredient in ALL_INGREDIENTS:
    if "Rice".lower() not in ingredient.lower():
        continue
    group = grouper.get_similar_ingredients(ingredient)
    if len(group) == 0:
        fehler.append(ingredient)
    else:
        normal.append((ingredient, group))
        
normal.sort(key=lambda x: len(x[1]), reverse=True)

for list_, name in [(fehler, "FEHLER"), (normal, "NORMAL")]:
    print(f"\n--- {name} ---")
    if list_:
        for item in list_:
            print(item)


--- FEHLER ---

--- NORMAL ---
('Basmati Rice', ['Basmati Rice'])
('Brown Rice', ['Brown Rice'])
('Jasmine Rice', ['Jasmine Rice'])
('Rice', ['Rice'])
('Rice Noodles', ['Rice Noodles'])
('Rice Stick Noodles', ['Rice Stick Noodles'])
('Rice Vermicelli', ['Rice Vermicelli'])
('Paella Rice', ['Paella Rice'])
('Rice Vinegar', ['Rice Vinegar'])
('Rice wine', ['Rice wine'])
('Rice Krispies', ['Rice Krispies'])
('Sushi Rice', ['Sushi Rice'])
('Flat Rice Noodles', ['Flat Rice Noodles'])
('Vermicelli Rice Noodles', ['Vermicelli Rice Noodles'])
('Brown Rice Noodle', ['Brown Rice Noodle'])
('Arabic Rice', ['Arabic Rice'])
('Rice Paper Sheets', ['Rice Paper Sheets'])
('Rice Flour Pancakes', ['Rice Flour Pancakes'])
('Seasoned Rice Vinegar', ['Seasoned Rice Vinegar'])


In [ ]:
def test(query):
    res = grouper.get_similar_ingredients(query)
    print(f"Ergebnis '{query}': {res}")     

test_pairs = [
    ("Walnuts", "Walnut", True),           # Der Fall, der dich geärgert hat
    ("Walnut", "Walnuts", True),           # Andersrum
    ("Tomatoes", "Tomato", True),          # Der Klassiker
    ("Basil Leaves", "Basil", True),       # 'leaf' ist nicht kritisch
    ("Chicken", "Chicken Liver", False),   # 'liver' ist kritisch -> Block
    ("Peanut", "Peanut Butter", False),    # 'butter' ist kritisch -> Block
    ("Apple", "Apple Vinegar", False) ,     # Kein kritisches Wort im Diff (Vinegar fehlt in Liste)
                                           # Falls du Essig nicht willst, pack 'vinegar' in die CRITICAL Liste!
    ("Cashew", "Cashew Nuts", True),            # True
    ("Basil", "Basil Leaves", True),            # True
    ("Chicken", "Chicken Liver", False),        # False
    ("Chili Powder","Curry Powder", False),     # False
    ("Duck Liver", "Chicken Liver", False),     # False
    ("Peach Juice","Lime Juice", False),        # False
    ("Peach Juice","Peach", False),             # False
    ("Peach","Peach Juice", False),             # False
    ("Peanut", "Peanut Butter", False),         # False (wegen Butter)
    ("Poppy Seeds","Cumin Seeds", False),       # False
    ("Tomato", "Plum Tomato", True),            # True
    ("Tomato", "Tomatoes", True),               # True
    ("Tomato", "Tomato Sauce", False)           # True
]
for input_ingredient, similar_ingredient, expected in test_pairs:
    if not (input_ingredient in ALL_INGREDIENTS) or not (similar_ingredient in ALL_INGREDIENTS):
        print(f"Skipping test for '{input_ingredient}' vs '{similar_ingredient}' as one or both are not in ALL_INGREDIENTS.")
        continue  # Skippe Tests, wenn beide Zutaten nicht in der DB sind
    result = grouper.get_similar_ingredients(input_ingredient)
    status = "✅" if (similar_ingredient in result) == expected else "❌"
    print(f"'{input_ingredient}' vs '{similar_ingredient}': {'Found' if similar_ingredient in result else 'Not Found'} (Expected: {expected}) {status}")

